# Relative PSF-Flux Light Curves — all DIA objects grouped by DDF field (src + fp)

This notebook is derived from `04c_relativeFlux_withfp.ipynb`.
Instead of one figure per DIA object, **one figure is produced per DDF field**,
with all DIA objects belonging to that field superimposed on the same subplots.

### Visual encoding
| Dimension | Encoding |
|---|---|
| DIA object identity | **colour** (discrete palette, one colour per object) |
| Photometric band    | **marker shape** (one shape per band) |
| Data type           | **fill**: src = filled marker, fp = hollow marker (`mfc="none"`) |

### Marker-shape convention
| Band | Marker |
|---|---|
| u | circle `o` |
| g | square `s` |
| r | triangle up `^` |
| i | diamond `D` |
| z | triangle down `v` |
| y | plus (filled) `P` |

### Colour palette
`matplotlib.colormaps['tab20']` + `'tab20b'` concatenated → 40 distinct colours.
If more than 40 objects appear in a field, colours wrap around (unlikely in practice).

### Normalisation convention
For each *(diaObjectId, band)*: normalisation factor = **median of src psfFlux**.
fp points are divided by the same median. Bands with no src are skipped.

### Subplot layout
`u | g | r | i | z | y | all-bands`  (7 panels, one figure per DDF field).

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Last update:** 2026-04-02  
- **Subject:** Fink alert broker — Rubin LSST photometric calibration check (all LC per DDF)

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_04d"
DIR_DATA_SRC = "data_FINK_BLOCK_LC_03"  # src alert data (from notebook 03)
DIR_DATA_FP = "data_FINK_BLOCK_LC_01"  # forced photometry data (from notebook 01)
DIR_FIGS = f"figs_{NB_TAG}"  # output figures
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Source files (src) ────────────────────────────────────────────────────────
FILE_BUTLER = os.path.join(DIR_DATA_SRC, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_SRC, "src_joined_consdb.parquet")

# ── Forced photometry file ────────────────────────────────────────────────────
FILE_FP = os.path.join(DIR_DATA_FP, "gaia_star_stable_fp.parquet")

# ── Band definitions ──────────────────────────────────────────────────────────
BANDS = list("ugrizy")

# Marker shape per band (same shape for src and fp; fill distinguishes them)
BAND_MARKERS = {
    "u": "o",  # circle
    "g": "s",  # square
    "r": "^",  # triangle up
    "i": "D",  # diamond
    "z": "v",  # triangle down
    "y": "P",  # plus (filled)
}

# ── Discrete colour palette for DIA objects (up to 40 colours) ───────────────
# tab20 (20) + tab20b (20) = 40 visually distinct colours
_cmap_a = mcm.get_cmap("tab20")
_cmap_b = mcm.get_cmap("tab20b")
OBJECT_PALETTE = [_cmap_a(i) for i in np.linspace(0, 1, 20, endpoint=False)] + [
    _cmap_b(i) for i in np.linspace(0, 1, 20, endpoint=False)
]

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Column names in the src parquet ──────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_FIELD = "field"

# ── Column names in the fp parquet (auto-detected in section 4) ──────────────
FP_COL_OBJ = "diaObjectId"  # placeholder — overwritten below
FP_COL_MJD = "midpointMjdTai"
FP_COL_BAND = "band"
FP_COL_PSF = "psfFlux"
FP_COL_PSFERR = "psfFluxErr"


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Src input directory : {os.path.abspath(DIR_DATA_SRC)}")
print(f"FP  input directory : {os.path.abspath(DIR_DATA_FP)}")
print(f"Figure directory    : {os.path.abspath(DIR_FIGS)}")
print(f"Object palette size : {len(OBJECT_PALETTE)} colours")

## 2. Load src data (alert stream)

In [ ]:
# Butler preferred; fallback to consDb
if os.path.exists(FILE_BUTLER):
    df_src = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file : {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df_src = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file : {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found. "
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

df_src[COL_MJD] = df_src[COL_MJD].astype(float)
print(f"Shape  : {df_src.shape[0]:,} rows × {df_src.shape[1]} columns")
print(f"Label  : {src_label}")

## 3. Load forced photometry data

In [ ]:
if not os.path.exists(FILE_FP):
    raise FileNotFoundError(
        f"Forced photometry file not found: {FILE_FP}\n"
        "Run notebook 01 to produce gaia_star_stable_fp.parquet first."
    )

df_fp_all = pd.read_parquet(FILE_FP)
print(f"Forced photometry file : {FILE_FP}")
print(f"Shape : {df_fp_all.shape[0]:,} rows × {df_fp_all.shape[1]} columns")

## 4. Schema inspection & column auto-detection

In [ ]:
print("=== SRC columns ===")
print(df_src.dtypes.to_string())

In [ ]:
print("=== FP columns ===")
print(df_fp_all.dtypes.to_string())

In [ ]:
# Validate required src columns
required_src = [COL_OBJ, COL_MJD, COL_BAND, COL_PSF, COL_PSFERR, COL_FIELD]
missing_src = [c for c in required_src if c not in df_src.columns]
if missing_src:
    raise KeyError(f"Missing required src columns: {missing_src}")
print("All required src columns present.")

# Auto-detect fp column names
fp_cols = df_fp_all.columns.tolist()


def _find_col(candidates, available, label):
    for c in candidates:
        if c in available:
            return c
    raise KeyError(
        f"Cannot find {label} column in fp file. Candidates tried: {candidates}. Available: {available[:20]}"
    )


FP_COL_OBJ = _find_col(["diaObjectId", "r:diaObjectId", "objectId"], fp_cols, "diaObjectId")
FP_COL_MJD = _find_col(["midpointMjdTai", "r:midpointMjdTai", "mjd", "MJD"], fp_cols, "midpointMjdTai")
FP_COL_BAND = _find_col(["band", "r:band", "filter"], fp_cols, "band")
FP_COL_PSF = _find_col(["psfFlux", "r:psfFlux"], fp_cols, "psfFlux")
FP_COL_PSFERR = _find_col(["psfFluxErr", "r:psfFluxErr"], fp_cols, "psfFluxErr")

df_fp_all[FP_COL_MJD] = df_fp_all[FP_COL_MJD].astype(float)

print("FP column mapping:")
for lbl, col in [
    ("diaObjectId", FP_COL_OBJ),
    ("midpointMjdTai", FP_COL_MJD),
    ("band", FP_COL_BAND),
    ("psfFlux", FP_COL_PSF),
    ("psfFluxErr", FP_COL_PSFERR),
]:
    print(f"  {lbl:18s} → {col}")

## 5. Build per-field DIA-object index

In [ ]:
# List DDF fields present in the src table
fields = sorted(df_src[COL_FIELD].dropna().unique())
print(f"DDF fields found : {fields}")

# For each field: sorted list of diaObjectIds (by descending alert count)
field_objects = {}
for field in fields:
    df_f = df_src[df_src[COL_FIELD] == field]
    counts = df_f.groupby(COL_OBJ).size().sort_values(ascending=False)
    field_objects[field] = counts.index.tolist()
    print(f"  {field:30s} : {len(counts):3d} objects  ({counts.sum():,} src alerts)")

## 6. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """Convert an array of MJD floats to ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def compute_src_median_and_ratio(flux, flux_err):
    """
    Normalise flux by median(flux) and propagate errors.

    Returns
    -------
    ratio, ratio_err, f_med, sigma_ratio
    """
    f = np.asarray(flux, dtype=float)
    fe = np.asarray(flux_err, dtype=float)

    finite_mask = np.isfinite(f) & (f > 0)
    if finite_mask.sum() == 0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), np.nan, np.nan

    f_med = float(np.median(f[finite_mask]))
    if f_med == 0.0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), 0.0, np.nan

    ratio = f / f_med
    ratio_err = fe / f_med
    sigma_ratio = float(np.nanstd(ratio[finite_mask]))
    return ratio, ratio_err, f_med, sigma_ratio


def normalise_fp(fp_flux, fp_flux_err, f_med_src):
    """Normalise fp fluxes using the src-derived median."""
    f = np.asarray(fp_flux, dtype=float)
    fe = np.asarray(fp_flux_err, dtype=float)
    if np.isnan(f_med_src) or f_med_src == 0.0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan)
    return f / f_med_src, fe / f_med_src


def _add_date_axis(ax, dt_array, t0_mjd):
    """Add a secondary x-axis on top of *ax* showing calendar dates."""
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


def get_object_color(obj_id, obj_list):
    """Return the colour assigned to obj_id within obj_list (index-based)."""
    idx = obj_list.index(obj_id) if obj_id in obj_list else 0
    return OBJECT_PALETTE[idx % len(OBJECT_PALETTE)]


def fp_obj_id_cast(obj_id, fp_series):
    """Cast obj_id to the dtype of the fp id column for safe comparison."""
    try:
        return type(fp_series.iloc[0])(obj_id)
    except Exception:
        return obj_id


print("Utility functions defined.")

## 7. Main plotting function: one figure per DDF field

Each figure has 7 subplots (`u | g | r | i | z | y | all-bands`).
All DIA objects of the field are superimposed:
- **colour** encodes the object
- **marker shape** encodes the band (same shape in individual-band subplot and in the combined panel)
- **filled** = src, **hollow** (`mfc="none"`) = fp

In [ ]:
def plot_field_all_objects(field, obj_list, df_src_field, df_fp_field, axes, yscale_min=0.5, yscale_max=1.5):
    """
    Plot normalised PSF-flux light curves for all DIA objects of one DDF field.

    Parameters
    ----------
    field         : str        — DDF field name
    obj_list      : list       — diaObjectIds in this field (ordered for colour assignment)
    df_src_field  : DataFrame  — src rows for all objects in this field
    df_fp_field   : DataFrame  — fp rows for all objects in this field (may be empty)
    axes          : list of 7 Axes — [u, g, r, i, z, y, all-bands]

    Returns
    -------
    t0_mjd_field : float — global MJD reference (minimum over all src in the field)
    """
    # Global t0 for the field = earliest src alert across all objects
    t0_mjd = df_src_field[COL_MJD].min()

    # Collect all dt values for the date-axis span (src + fp)
    all_dt = np.concatenate(
        [
            df_src_field[COL_MJD].values - t0_mjd,
            (df_fp_field[FP_COL_MJD].values - t0_mjd) if len(df_fp_field) > 0 else np.array([]),
        ]
    )

    # Iterate over objects
    # Legend entries: one proxy per object (colour swatch) + one per band (marker shape)
    # We accumulate them and add to the combined panel only
    obj_legend_handles = []  # colour proxies
    band_legend_handles = []  # shape proxies
    band_shapes_added = set()

    for obj_id in obj_list:
        color = get_object_color(obj_id, obj_list)

        # --- fp rows for this object ---
        fp_obj_cast = fp_obj_id_cast(obj_id, df_fp_field[FP_COL_OBJ]) if len(df_fp_field) > 0 else obj_id
        df_fp_obj = (
            df_fp_field[df_fp_field[FP_COL_OBJ] == fp_obj_cast] if len(df_fp_field) > 0 else pd.DataFrame()
        )

        # --- src rows for this object ---
        df_src_obj = df_src_field[df_src_field[COL_OBJ] == obj_id].sort_values(COL_MJD)

        # ── Per-band subplots ──────────────────────────────────────────────────
        for idx, band in enumerate(BANDS):
            ax = axes[idx]
            marker = BAND_MARKERS[band]

            mask_src = df_src_obj[COL_BAND] == band
            df_b_src = df_src_obj[mask_src]

            if len(df_b_src) == 0:
                continue  # no src in this band for this object → skip

            dt_src = df_b_src[COL_MJD].values - t0_mjd
            src_ratio, src_ratio_err, f_med, sigma_src = compute_src_median_and_ratio(
                df_b_src[COL_PSF].values, df_b_src[COL_PSFERR].values
            )

            # src — filled
            ax.errorbar(
                dt_src,
                src_ratio,
                yerr=src_ratio_err,
                fmt=marker,
                ms=4,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.80,
            )

            # fp — hollow
            if len(df_fp_obj) > 0 and not np.isnan(f_med):
                mask_fp = df_fp_obj[FP_COL_BAND] == band
                df_b_fp = df_fp_obj[mask_fp].sort_values(FP_COL_MJD)
                if len(df_b_fp) > 0:
                    dt_fp = df_b_fp[FP_COL_MJD].values - t0_mjd
                    fp_ratio, fp_ratio_err = normalise_fp(
                        df_b_fp[FP_COL_PSF].values,
                        df_b_fp[FP_COL_PSFERR].values,
                        f_med,
                    )
                    ax.errorbar(
                        dt_fp,
                        fp_ratio,
                        yerr=fp_ratio_err,
                        fmt=marker,
                        ms=5,
                        color="none",
                        mec=color,
                        mew=1.2,
                        ecolor=color,
                        elinewidth=0.7,
                        capsize=2,
                        alpha=0.75,
                    )

        # ── Combined panel (subplot 7) — same object, all bands ───────────────
        ax_all = axes[-1]
        first_band_for_obj = True

        for band in BANDS:
            marker = BAND_MARKERS[band]
            mask_src = df_src_obj[COL_BAND] == band
            df_b_src = df_src_obj[mask_src]

            if len(df_b_src) == 0:
                continue

            dt_src = df_b_src[COL_MJD].values - t0_mjd
            src_ratio, src_ratio_err, f_med, _ = compute_src_median_and_ratio(
                df_b_src[COL_PSF].values, df_b_src[COL_PSFERR].values
            )

            # src
            h_src = ax_all.errorbar(
                dt_src,
                src_ratio,
                yerr=src_ratio_err,
                fmt=marker,
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.6,
                capsize=2,
                alpha=0.75,
            )

            # fp
            if len(df_fp_obj) > 0 and not np.isnan(f_med):
                mask_fp = df_fp_obj[FP_COL_BAND] == band
                df_b_fp = df_fp_obj[mask_fp].sort_values(FP_COL_MJD)
                if len(df_b_fp) > 0:
                    dt_fp = df_b_fp[FP_COL_MJD].values - t0_mjd
                    fp_ratio, fp_ratio_err = normalise_fp(
                        df_b_fp[FP_COL_PSF].values,
                        df_b_fp[FP_COL_PSFERR].values,
                        f_med,
                    )
                    ax_all.errorbar(
                        dt_fp,
                        fp_ratio,
                        yerr=fp_ratio_err,
                        fmt=marker,
                        ms=4,
                        color="none",
                        mec=color,
                        mew=1.2,
                        ecolor=color,
                        elinewidth=0.6,
                        capsize=2,
                        alpha=0.70,
                    )

            # Accumulate band legend proxy (once per band)
            if band not in band_shapes_added:
                band_shapes_added.add(band)
                proxy_src = plt.Line2D(
                    [],
                    [],
                    linestyle="",
                    marker=marker,
                    ms=5,
                    color="k",
                    label=f"{band} src (●=filled)",
                )
                proxy_fp = plt.Line2D(
                    [],
                    [],
                    linestyle="",
                    marker=marker,
                    ms=5,
                    color="k",
                    mfc="none",
                    mew=1.2,
                    label=f"{band} fp  (○=hollow)",
                )
                band_legend_handles.extend([proxy_src, proxy_fp])

        # Object colour proxy (from first band plotted)
        obj_proxy = plt.Line2D(
            [],
            [],
            linestyle="",
            marker="o",
            ms=5,
            color=color,
            label=f"obj {obj_id}",
        )
        obj_legend_handles.append(obj_proxy)

    # ── Finalise per-band subplots ─────────────────────────────────────────────
    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"band {band}  [{BAND_MARKERS[band]}]", fontsize=9)
        ax.set_ylabel("psfFlux / median(src)", fontsize=8)
        _add_date_axis(ax, all_dt, t0_mjd)

    # ── Finalise combined panel ────────────────────────────────────────────────
    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands  |  src (filled) + fp (hollow)", fontsize=9)
    ax_all.set_ylabel("psfFlux / median(src)", fontsize=8)

    # Two separate legends: objects (colours) + bands (shapes)
    leg_bands = ax_all.legend(
        handles=band_legend_handles,
        fontsize=6,
        ncol=2,
        loc="upper right",
        title="band (shape)",
        title_fontsize=6,
        framealpha=0.6,
    )
    ax_all.add_artist(leg_bands)  # keep first legend when adding second
    ax_all.legend(
        handles=obj_legend_handles,
        fontsize=5,
        ncol=2,
        loc="upper left",
        title="diaObject (colour)",
        title_fontsize=6,
        framealpha=0.6,
    )
    _add_date_axis(ax_all, all_dt, t0_mjd)

    return t0_mjd


print("plot_field_all_objects() defined.")

## 8. Main loop: one figure per DDF field

In [ ]:
n_subplots = len(BANDS) + 1  # 6 bands + combined

for field in fields:
    obj_list = field_objects[field]
    n_obj = len(obj_list)

    # src rows for this field
    df_src_field = df_src[df_src[COL_FIELD] == field].copy()

    # fp rows for this field — match object ids present in src
    if len(df_fp_all) > 0:
        # Cast obj_list to fp id dtype for safe filtering
        try:
            fp_ids = [type(df_fp_all[FP_COL_OBJ].iloc[0])(x) for x in obj_list]
        except Exception:
            fp_ids = obj_list
        df_fp_field = df_fp_all[df_fp_all[FP_COL_OBJ].isin(fp_ids)].copy()
    else:
        df_fp_field = pd.DataFrame()

    n_src_field = len(df_src_field)
    n_fp_field = len(df_fp_field)

    print(f"\n── Field: {field}  |  {n_obj} objects  |  {n_src_field:,} src  |  {n_fp_field:,} fp ──")

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 5.0),
        sharey=False,
        constrained_layout=True,
    )

    t0_mjd = plot_field_all_objects(field, obj_list, df_src_field, df_fp_field, axes)
    t0_date = mjd_to_datestr([t0_mjd])[0]

    # Common bottom x-label
    for ax in axes:
        ax.set_xlabel("Δt [days] from first src alert in field", fontsize=8)

    fig.suptitle(
        f"DDF field: {field}  |  {n_obj} DIA objects  |  {n_src_field:,} src  {n_fp_field:,} fp  |  t₀={t0_date}\n"
        "colour = diaObject   |   shape = band   |   filled = src   |   hollow = fp",
        fontsize=10,
        y=1.05,
    )

    # Sanitise field name for filename
    field_safe = field.replace(" ", "_").replace("/", "-")
    savefig(f"relflux_psf_srcfp_DDF_{field_safe}_{src_label}")
    plt.show()
    # plt.close(fig)

## 9. Summary table: per-field, per-object, per-band statistics

In [ ]:
records = []

for field in fields:
    obj_list = field_objects[field]
    try:
        fp_ids = [type(df_fp_all[FP_COL_OBJ].iloc[0])(x) for x in obj_list]
    except Exception:
        fp_ids = obj_list
    df_fp_field = df_fp_all[df_fp_all[FP_COL_OBJ].isin(fp_ids)] if len(df_fp_all) > 0 else pd.DataFrame()

    for rank_in_field, obj_id in enumerate(obj_list):
        df_src_obj = df_src[(df_src[COL_FIELD] == field) & (df_src[COL_OBJ] == obj_id)]
        fp_cast = fp_obj_id_cast(obj_id, df_fp_field[FP_COL_OBJ]) if len(df_fp_field) > 0 else obj_id
        df_fp_obj = (
            df_fp_field[df_fp_field[FP_COL_OBJ] == fp_cast] if len(df_fp_field) > 0 else pd.DataFrame()
        )

        t0_mjd = df_src_obj[COL_MJD].min() if len(df_src_obj) > 0 else np.nan
        t0_date = mjd_to_datestr([t0_mjd])[0] if not np.isnan(t0_mjd) else "N/A"

        row = {
            "field": field,
            "rank_in_field": rank_in_field + 1,
            "diaObjectId": obj_id,
            "n_src": len(df_src_obj),
            "n_fp": len(df_fp_obj),
            "t0_date": t0_date,
        }

        for band in BANDS:
            df_b = df_src_obj[df_src_obj[COL_BAND] == band]
            if len(df_b) == 0:
                row[f"sigma_src_{band}"] = np.nan
                row[f"n_fp_{band}"] = 0
            else:
                _, _, _, sigma = compute_src_median_and_ratio(df_b[COL_PSF].values, df_b[COL_PSFERR].values)
                row[f"sigma_src_{band}"] = round(sigma, 4)
                row[f"n_fp_{band}"] = int((df_fp_obj[FP_COL_BAND] == band).sum()) if len(df_fp_obj) > 0 else 0

        records.append(row)

df_summary = pd.DataFrame(records)
print("Summary table — σ(src ratio) and fp counts per field / object / band:")
display(df_summary)

In [ ]:
summary_path = os.path.join(DIR_FIGS, f"sigma_ratio_srcfp_by_field_{src_label}.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")